# M4 · Dane tabelaryczne: SQL, Genie Agent i kontrola dostępu

**"Ile mamy klientów VIP?" Dwie poprawne odpowiedzi:**

| Z dokumentów (RAG, M3) | Z tabeli (funkcja albo Genie) |
|---|---|
| "Według raportu o segmentach klienci VIP to około 9,5 tys. firm z najniższym recency... [01_segmentacja_klientow #1]" | **9 541** wierszy, czyli **9 494** klientów |
| narracja, stan z dnia wygenerowania raportu, kontekst i wnioski | liczba, stan na teraz, zero interpretacji |

Pytanie "ile" idzie do tabeli, a pytanie "dlaczego" do dokumentów. Agent musi to rozróżniać i uczymy go tego opisami narzędzi.

**Trzy drogi do tej samej tabeli:**
- **SQL wprost:** pełna swoboda, zero kontroli. Nadaje się dla człowieka, nie dla agenta.
- **Funkcja Unity Catalog** (M2): stałe pytanie, stały kształt odpowiedzi, bez PII. Kontrakt między danymi a agentem.
- **Genie Agent** (dawniej Genie Space): zamienia pytanie po polsku na SQL i zwraca wynik. Do pytań ad hoc, których nie przewidziały funkcje.

| Część | Co robisz | Lab |
|---|---|---|
| 1 | oczekiwane wyniki czterech pytań z SQL | 2 z 4 zapytań |
| 2 | Genie Agent w UI i porównanie z RAG | UI |
| 3 | uprawnienia, row filter (demo), column mask | wyrażenie maski |
| 4 | **obowiązkowe** zdjęcie filtra i maski przed M5 | brak |

## Mapa ścieżek

| Ścieżka | Co robisz | Gotowe, gdy | Sekcja |
|---|---|---|---|
| **A · Razem** (TechRetail) | oczekiwane wyniki z SQL, Genie Agent nad tabelą Gold, row filter i column mask, obowiązkowe zdjęcie polityk | Genie zgadza się z tabelą oczekiwanych wyników, a `m4-cleanup` drukuje 28 813 wierszy | części 1-4 |
| **B · Samodzielnie** (Bakehouse) | Genie Agent na transakcjach i franczyzach piekarni, maska na numerze karty | Genie podaje te same liczby co SQL, maska ukrywa wszystkie numery kart i jest zdjęta | "B · Samodzielnie: Genie Agent na piekarni i maska numeru karty" |
| **C · Wyzwanie** (Airbnb) | row filter i column mask na jednej tabeli, pytanie do Genie przy nałożonych politykach | przy politykach widać tylko Mission i `***` w `name`, po zdjęciu znów 8 533 oferty | "C · Wyzwanie: dwie polityki na jednej tabeli" |

Ścieżka A to pełny cel modułu. B i C robisz, gdy skończysz A.

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
dbutils.library.restartPython()

**Infrastruktura.** Konfiguracja wspólna dla wszystkich modułów. Uruchom i czytaj dalej, tu nie ma nic do nauczenia.


In [ ]:
# Wspólna konfiguracja warsztatu. Ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
BH_SCHEMA = "bakehouse"       # ścieżka B: kopie danych Bakehouse i funkcje-narzędzia
AIRBNB_SCHEMA = "airbnb"      # ścieżka C: oferty Airbnb i funkcje-narzędzia
POLICY_SCHEMA = "governance"  # maski i filtry ścieżek B i C: poza schematami, które MCP wystawia agentowi
BH_TRANSACTIONS = f"{CATALOG}.{BH_SCHEMA}.transactions"
BH_REVIEWS = f"{CATALOG}.{BH_SCHEMA}.reviews"
AIRBNB_TABLE = f"{CATALOG}.{AIRBNB_SCHEMA}.listings"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

import logging
# MLflow w notebooku serverless (UI) wypisuje przy tracingu stos Py4JSecurityException z "resolving tags".
# To ostrzeżenie, nie błąd. Trace zapisuje się poprawnie, a wyciszamy je, żeby nikt nie wziął go za błąd.
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

**Infrastruktura.** Komórka przygotowuje moduł. Uruchom ją i czytaj dalej.

Oprócz połączenia z Databricks definiuje funkcję `run_statements()`. Dostaje listę poleceń SQL i wykonuje je po kolei. Gdy któreś się nie uda, wypisuje błąd i przechodzi do następnego, zamiast przerywać. Korzysta z niej sprzątanie w części 4. Jeśli filtr był już zdjęty, jego `DROP` się nie uda, ale maska i tak zostanie zdjęta. Na końcu komórka wypisuje liczbę wierszy tabeli Gold.


In [ ]:
import re
import time

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
USERNAME = spark.sql("SELECT current_user()").first()[0]


def run_statements(statements: list) -> None:
    """Wykonuje polecenia po kolei i nie przerywa na błędzie: sprzątanie ma dojść do końca.

    DROP polityki, której nie ma, kończy się błędem, a zostawiona polityka psuje kolejne moduły.
    """
    for statement in statements:
        try:
            spark.sql(statement)
            print(f"OK  {statement}")
        except Exception as error:
            print(f"--  {statement}: {str(error)[:120]}")


rows = spark.table(GOLD_TABLE).count()
print(f"Użytkownik: {USERNAME} | tabela: {GOLD_TABLE} ({rows:,} wierszy)")


## 1. Odpowiedź z tabeli: oczekiwane wyniki

> **Cel:** mieć własną odpowiedź, zanim zapytasz Genie.
> **Gotowe, gdy:** masz pięć liczb policzonych SQL-em i zapisanych.


Zanim zapytasz Genie, ustal, **co jest poprawną odpowiedzią**. Te same liczby są w decku, w Przewodniku i w scorerach ewaluacji. To pierwsze wiersze zestawu testowego, który w produkcji uruchamia się przed każdą zmianą.

Uwaga na dwie średnie: `AVG(monetary)` po wierszach daje **1038,72**, a po klientach (z `DISTINCT`, jak w funkcji z M2) **1043,15**. Ta sama przyczyna co 9 541 kontra 9 494: 143 klientów ma po dwa wiersze.

**Lab:** dopisz dwa brakujące zapytania. Komórka pomija te, które mają jeszcze `...`.

In [ ]:
# ZADANIE 10: dwa zapytania z oczekiwanym wynikiem.
EXPECTED_SQL = {
    "Ile mamy klientów VIP (loyalty_segment = 3)?":
        f"SELECT COUNT(*) AS vip FROM {GOLD_TABLE} WHERE loyalty_segment = 3",
    "Jaki stan ma najwięcej klientów?":
        f"""SELECT state, COUNT(*) AS klienci FROM {GOLD_TABLE}
            GROUP BY state ORDER BY klienci DESC LIMIT 1""",
    # TODO: liczba klientów z num_orders = 0
    "Ile klientów nie złożyło żadnego zamówienia?": ...,
    # TODO: średnia monetary w segmencie 3, zaokrąglona do 2 miejsc
    "Jaka jest średnia wartość monetary dla segmentu VIP?": ...,
    # Wiersz to nie klient: 143 klientów ma zdublowany wiersz, więc Genie poda 9 494,
    # a nie 9 541. Obie liczby są poprawne, różnią się definicją metryki.
    "Ile mamy klientów VIP: wierszy czy unikalnych klientów?":
        f"""SELECT COUNT(*) AS wiersze, COUNT(DISTINCT customer_id) AS klienci
            FROM {GOLD_TABLE} WHERE loyalty_segment = 3""",
}

expected_values = {}
for question, query in EXPECTED_SQL.items():
    if query is ...:
        print(f"--  {question}  (TODO)")
        continue
    expected_values[question] = spark.sql(query).first().asDict()
    print(f"OK  {question}: {expected_values[question]}")
# Oczekiwane: 9 541 wierszy VIP (9 494 klientów), NY 3 417, 26 862 bez zamówień, 1038.72


### Kiedy funkcja Unity Catalog, a kiedy Genie

> **Cel:** umieć wybrać między Genie a funkcją Unity Catalog.
> **Gotowe, gdy:** dla swojego pytania potrafisz powiedzieć, które z dwóch narzędzi jest właściwe i dlaczego.


| Kryterium | Genie Agent | Funkcja Unity Catalog |
|---|---|---|
| Styl zapytania | język naturalny zamieniany na SQL nad wskazanymi tabelami | parametry do gotowej, przewidywalnej logiki |
| Elastyczność | wysoka: pytania otwarte | celowo ograniczona |
| Najlepsze do | analityka eksploracyjna, pytania ad hoc | powtarzalne pobranie danych, logika biznesowa |
| Dostęp dla agenta | zarządzany serwer MCP `/api/2.0/mcp/genie/{id}` (M6) | `EXECUTE` na funkcji, test payloadem bez modelu |
| U nas dziś | UI i Tool w Playground | trzy funkcje w agencie (M2, M5) |

Z kursu Databricks: *komplementarne, nie konkurencyjne*. Supervisor pyta Genie "co napędzało wzrost w Q4?", a funkcję "pokaż klienta 1234". "Co mówią dokumenty" idzie do RAG.

## 2. Lab: Genie Agent nad tabelą Gold

> **Cel:** własny Genie Agent nad tabelą Gold.
> **Gotowe, gdy:** Genie odpowiada na cztery pytania i pokazuje wygenerowany SQL.


1. W lewym pasku **Genie Agents → New** (w nowym UI *Genie Agent*, dawniej *Genie Space*).
2. **Tytuł:** `Retail Customer Intelligence Assistant` (dokładnie tak, bo M6 i ewaluacja szukają tej nazwy). **Dane:** `workspace.default.gold_customer_360`. **Warehouse:** jedyny Serverless na Free Edition.
3. **Instructions:**
   ```text
   Odpowiadaj po polsku. Główna tabela: workspace.default.gold_customer_360.
   loyalty_segment: 0=Nowi/Nieaktywni, 1=Rozwijający się, 2=Regularni, 3=VIP.
   tax_id i customer_name to PII, nie pokazuj ich wartości.
   Kwoty (monetary, avg_item_value) są w USD.
   ```
4. **Sample questions:** "Ile mamy klientów VIP (loyalty_segment = 3)?", "Który stan ma najwięcej klientów i jaką średnią wartość monetary?", "Jaki procent klientów nie złożył żadnego zamówienia (num_orders = 0)?", "Porównaj średni recency_days i monetary między segmentami".
5. Zadaj Genie cztery pytania z części 1 i porównaj z oczekiwanymi wynikami. Rozwiń **Show code**, żeby zobaczyć wygenerowany SQL.
6. Porównaj z M3: to samo pytanie w RAG dało narrację z cytatem, a w Genie liczbę i SQL. Które kiedy wybrać?

| Pytanie | Oczekiwane (SQL) | Genie | RAG (M3) |
|---|---|---|---|
| VIP | 9 541 wierszy (9 494 klientów) | | |
| stan z największą liczbą klientów | NY, 3 417 | | |
| bez zamówień | 26 862 | | |
| średnia monetary VIP | 1038,72 | | |

**(opcjonalnie) Genie z kodu.** To samo co w interfejsie, tylko przez SDK.

- `find_genie_space(title)` szuka Genie Agenta po tytule i zwraca jego identyfikator. Dlatego tytuł musi być dokładnie taki jak w kroku 2.
- `ask_genie(space_id, question)` zadaje pytanie, czeka na odpowiedź i wypisuje wygenerowany SQL, pierwsze wiersze wyniku i komentarz Genie.

Komórka zadaje dwa pierwsze pytania z części 1. Między pytaniami czeka kilkanaście sekund, bo na Free Edition API Genie przyjmuje około pięciu pytań na minutę. Jeśli Genie Agenta jeszcze nie ma, komórka to wypisze i nic więcej nie zrobi.


In [ ]:
# Genie Agent z kodu: to samo pytanie co w UI, tylko przez SDK.
def find_genie_space(title: str) -> str | None:
    """Identyfikator Genie Agenta o danym tytule. None, gdy go nie ma albo API jest niedostępne."""
    try:
        spaces = w.genie.list_spaces().spaces or []
    except Exception as error:
        print(f"API Genie niedostępne ({type(error).__name__}): zostaje pokaz w interfejsie.")
        return None
    return next((space.space_id for space in spaces if (space.title or "") == title), None)


def ask_genie(space_id: str, question: str) -> None:
    """Zadaje pytanie Genie i wypisuje wygenerowany SQL, jego wynik i komentarz."""
    print(f"Pytanie: {question}")
    message = w.genie.start_conversation_and_wait(space_id=space_id, content=question)
    for attachment in message.attachments or []:
        if attachment.query:
            print(f"   SQL: {attachment.query.query}")
            result = w.genie.get_message_attachment_query_result(
                space_id, message.conversation_id, message.message_id, attachment.attachment_id)
            print(f"   wynik: {result.statement_response.result.data_array[:3]}")
        if attachment.text:
            print(f"   odpowiedź: {attachment.text.content}")


GENIE_SPACE_ID = find_genie_space(GENIE_TITLE)
if not GENIE_SPACE_ID:
    print(f"Nie znaleziono Genie Agenta '{GENIE_TITLE}'. Utwórz go w UI (krok 2 wyżej).")
else:
    for question in list(EXPECTED_SQL)[:2]:
        ask_genie(GENIE_SPACE_ID, question)
        time.sleep(13)  # limit API Genie na Free Edition: ok. 5 pytań na minutę


## 3. Dostęp do danych w kontrolowany sposób: dwie warstwy

> **Cel:** zobaczyć, że dostęp kontroluje się w danych, a nie w prompcie.
> **Gotowe, gdy:** rozróżniasz ochronę w narzędziu (M2) i ochronę w katalogu (teraz).


| W narzędziu (M2) | W danych (Unity Catalog, teraz) |
|---|---|
| funkcja zwraca tylko wybrane kolumny, bez `tax_id` | **column mask**: `tax_id` zamaskowany wszędzie: w SQL, w Genie i w funkcjach |
| agent nie dostaje tabeli, tylko funkcję | **row filter**: użytkownik widzi tylko swoje stany |
| `GRANT EXECUTE` na funkcji decyduje, kto ją wywoła | działa niezależnie od modelu, promptu i narzędzia |

**Obrona w głąb.** Prompt może zawieść i funkcja może zawieść, ale maska w katalogu nie. Dlatego compliance wymaga tej trzeciej warstwy.

**Uprawnienia (least privilege).** Minimum dla agenta z M5, gdy działa jako service principal:

```sql
GRANT EXECUTE ON FUNCTION workspace.default.get_average_customer_value TO `retail-agent-sp`;
GRANT EXECUTE ON FUNCTION workspace.default.get_customer_profile       TO `retail-agent-sp`;
GRANT EXECUTE ON FUNCTION workspace.default.format_customer_for_agent  TO `retail-agent-sp`;
GRANT SELECT  ON TABLE    workspace.default.retail_rag_chunks_index    TO `retail-agent-sp`;
-- Żadnego SELECT na gold_customer_360: agent dostaje funkcje, nie tabelę.
```

Na Free Edition jesteś jedynym użytkownikiem i właścicielem obiektów, więc `GRANT` niczego nie odbierze Tobie. Dlatego filtr i maskę poniżej piszemy z warunkiem opartym na **grupie, do której nie należysz**. Zobaczysz dokładnie to, co zobaczyłby analityk bez uprawnień.

> **Uwaga na grupy.** Grupy `all_states_analysts` i `compliance_officers` **nie istnieją na Free Edition** i nic tego nie zmieni: tworzenie grup wymaga konsoli konta. `is_account_group_member('nieistniejąca_grupa')` zwraca fałsz, więc polityka obejmuje **każdego**, łącznie z Tobą. Na warsztacie to jest zaleta, bo od razu widać efekt. W płatnym workspace grupy trzeba najpierw założyć (**Settings → Identity and access → Groups**), inaczej nałożysz politykę, której nikt nie przejdzie, łącznie z kontem serwisowym agenta.

In [ ]:
%sql
SHOW GRANTS ON TABLE workspace.default.gold_customer_360

### Row filter: które wiersze widzi użytkownik

> **Cel:** jedna polityka, która działa wszędzie: w notebooku, w Genie i w agencie.
> **Gotowe, gdy:** po nałożeniu filtra to samo zapytanie zwraca mniej wierszy, także w Genie.


> **Uwaga: od tej komórki tabela ma nałożoną politykę.** Jeśli przerwiesz moduł w dowolnym miejscu, bo utkniesz na zadaniu, skończy się czas albo odłączy compute, wróć na koniec notebooka i uruchom `m4-cleanup`. Z aktywnym filtrem M5 i M6 nie ruszą: ich pierwsza komórka przerywa z komunikatem o liczbie wierszy.

Row filter to funkcja SQL zwracająca `BOOLEAN`. Unity Catalog wstrzykuje ją do każdego zapytania na tabeli: z notebooka, z Genie, z dashboardu i z funkcji agenta. Wiersz jest widoczny tylko wtedy, gdy funkcja zwraca `TRUE`.

**Scenariusz:** analityk regionu Kalifornia widzi tylko klientów z CA, a pełny obraz ma grupa `all_states_analysts`. Warunek nie może sprawdzać grupy, do której należy każdy użytkownik, bo byłby zawsze prawdziwy i filtr niczego by nie ukrywał. Tu warunek zależy od grupy, której w Twoim workspace nie ma.

**Razem z prowadzącym:** uruchom tę komórkę i sprawdź wynik. Predykat jest gotowy, nie musisz nic uzupełniać.

In [ ]:
%sql
CREATE OR REPLACE FUNCTION workspace.default.retail_row_filter(state_val STRING)
RETURNS BOOLEAN
COMMENT 'Row filter warsztatu: wszystkie stany widzi grupa all_states_analysts, pozostali tylko CA.'
RETURN is_account_group_member('all_states_analysts') OR state_val = 'CA';

ALTER TABLE workspace.default.gold_customer_360
SET ROW FILTER workspace.default.retail_row_filter ON (state);

-- Test: zostaje tylko CA
SELECT state, COUNT(*) AS klienci
FROM workspace.default.gold_customer_360
GROUP BY state
ORDER BY klienci DESC;

### Column mask: jakie wartości kolumny widzi użytkownik

> **Cel:** ukryć wartość, a nie cały wiersz.
> **Gotowe, gdy:** `tax_id` pokazuje `***MASKED***`, a reszta wiersza jest nadal widoczna.


Row filter ukrywa całe wiersze, a column mask ukrywa wartość w kolumnie. Wiersz jest widoczny, ale `tax_id` pokazuje `***MASKED***`. Maska zwraca ten sam typ co kolumna i działa także w Genie.

**Czym zastąpić wartość: znacznikiem czy `NULL`?** Tu wybieramy widoczny znacznik `***MASKED***`, bo na sali ma być widać, że maska zadziałała, a nie że kolumna jest pusta. W produkcji decyzja idzie zwykle w drugą stronę i ścieżka B (maska na `cardNumber`) pokazuje ten wariant: `CAST(NULL AS ...)`. Powód jest audytowy. `NULL` nie zdradza ani tego, że wartość istniała, ani jej długości, i nie da się go pomylić z prawdziwymi danymi w eksporcie. Trzecia droga to maska częściowa (`***-**12345`), gdy analityk musi odróżniać rekordy, nie znając wartości. Wybierz jedną i trzymaj się jej w całej domenie.

**Lab:** napisz wyrażenie `CASE`: prawdziwą wartość widzi tylko grupa `compliance_officers`.

In [ ]:
%sql
-- ZADANIE 11: wyrażenie maski.
CREATE OR REPLACE FUNCTION workspace.default.mask_tax_id(tax_id_val STRING)
RETURNS STRING
COMMENT 'Column mask warsztatu: prawdziwy tax_id widzi tylko grupa compliance_officers.'
-- TODO: CASE WHEN <członek grupy compliance_officers> THEN <wartość> ELSE '***MASKED***' END
RETURN TODO;

ALTER TABLE workspace.default.gold_customer_360
ALTER COLUMN tax_id SET MASK workspace.default.mask_tax_id;

-- Test: filtr i maska działają razem (tylko CA, tax_id zamaskowany)
SELECT customer_id, state, loyalty_segment, tax_id
FROM workspace.default.gold_customer_360
WHERE tax_id IS NOT NULL
LIMIT 5;

**Sprawdź w Genie:** zapytaj swojego Genie Agenta "Pokaż tax_id pięciu klientów" i "Ilu mamy klientów w NY?". Genie wykonuje SQL z Twoją tożsamością, więc filtr i maska obowiązują także tam: zobaczysz `***MASKED***` i 0 klientów w NY.

To jest sedno warstwy danych: nie trzeba nic zmieniać w Genie, w funkcjach ani w promptach.

> **Demo prowadzącego (Premium).** Prowadzący pokazuje filtr i maskę z dwóch kont na Premium: członek grup `all_states_analysts` i `compliance_officers` widzi wszystko, a konto bez tych grup widzi tylko CA i `***MASKED***`. Na Free Edition jesteś jedyną tożsamością, więc widzisz wariant "bez uprawnień".


## 4. Obowiązkowo przed M5: zdejmij filtr i maskę

> **Cel:** wrócić do pełnych danych przed M5.
> **Gotowe, gdy:** tabela ma 28 813 wierszy, a `tax_id` jest odsłonięte.


Ta komórka jest obowiązkowa także wtedy, gdy nie skończyłeś zadań wyżej: polityka raz nałożona zostaje na tabeli, dopóki jej nie zdejmiesz.

Agent z M5 ma odpowiadać o wszystkich stanach. Z aktywnym filtrem widziałby tylko CA, a macierz tras dałaby fałszywe wyniki. Komórka poniżej zdejmuje filtr i maskę, usuwa funkcje i sprawdza, że tabela wróciła do 28 813 wierszy. Polecenia wykonuje `run_statements()`, więc błąd jednego z nich nie zatrzymuje reszty. Na końcu dwie asercje sprawdzają liczbę wierszy i to, że `tax_id` znów wygląda jak numer, a nie `***MASKED***`.

W produkcji nie zdejmuje się zabezpieczeń bez przeglądu: `SHOW GRANTS`, `DESCRIBE TABLE EXTENDED` (sekcje *Row Filter* i *Column Masks*) oraz `information_schema`.

In [ ]:
run_statements([
    f"ALTER TABLE {GOLD_TABLE} DROP ROW FILTER",
    f"ALTER TABLE {GOLD_TABLE} ALTER COLUMN tax_id DROP MASK",
    f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.retail_row_filter",
    f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.mask_tax_id",
])

rows = spark.table(GOLD_TABLE).count()
sample_tax_id = spark.table(GOLD_TABLE).where("tax_id IS NOT NULL").select("tax_id").first()[0]
assert rows == 28_813, f"Tabela ma {rows} wierszy, row filter jest nadal aktywny?"
assert re.fullmatch(r"\d{2}-\d{7}", sample_tax_id), "tax_id nadal zamaskowany, sprawdź DROP MASK"
print(f"\nGotowe: {GOLD_TABLE} ma {rows:,} wierszy, tax_id bez maski. Możesz przejść do M5.")


### (jeśli zostanie czas) Genie Agent jako Tool w Playground

**Playground → Tools → Add tool → Genie**, wybierz `Retail Customer Intelligence Assistant`. Dodaj obok funkcje z M2. Zapytaj o coś, czego funkcje nie przewidziały, np. *"Który stan ma najwyższą średnią wartość klienta w segmencie Regularni?"*. Które narzędzie wybrał model? Rozwiń panel i zobacz SQL wygenerowany przez Genie.

## B · Samodzielnie: Genie Agent na piekarni i maska numeru karty (Bakehouse)

> **Cel:** Genie Agent, który odpowiada o sprzedaży sieci piekarni tak samo jak SQL, i maska na `cardNumber`.
> **Lekcja:** Genie potrzebuje warstwy semantycznej (instrukcji i złączeń), a numer karty chroni dopiero polityka w katalogu, nie instrukcja.
> **Gotowe, gdy:** Genie podaje te same cztery wyniki co komórka poniżej, a komórka z maską drukuje 0 widocznych numerów kart i potwierdza zdjęcie maski.

Pracujesz na kopiach z `00_setup`: `workspace.bakehouse.transactions` (transakcje z kolumną `cardNumber`) i `workspace.bakehouse.franchises` (punkty sieci). Tak jak w części 1, najpierw liczysz odpowiedzi SQL-em, a dopiero potem pytasz Genie. Nazwa punktu jest w tabeli franczyz, a sprzedaż w transakcjach, więc Genie musi złączyć obie tabele po `franchiseID`.

In [ ]:
FRANCHISES_TABLE = f"{CATALOG}.{BH_SCHEMA}.franchises"
BH_EXPECTED_SQL = {
    "Która franczyza ma największą sprzedaż?":
        f"""SELECT f.name, t.franchiseID, ROUND(SUM(t.totalPrice), 2) AS sprzedaz
            FROM {BH_TRANSACTIONS} t JOIN {FRANCHISES_TABLE} f ON f.franchiseID = t.franchiseID
            GROUP BY f.name, t.franchiseID ORDER BY sprzedaz DESC, t.franchiseID LIMIT 1""",
    # TODO: liczba wszystkich transakcji w BH_TRANSACTIONS
    "Ile jest transakcji?": ...,
    # TODO: metoda płatności (paymentMethod) z największą liczbą transakcji;
    #       przy remisie ORDER BY także po nazwie
    "Jaka metoda płatności jest najpopularniejsza?": ...,
    # TODO: produkt z największą sumą quantity. "Najlepiej sprzedający się" to definicja:
    #       zapisz ją też w instrukcjach Genie.
    "Który produkt sprzedaje się najlepiej (liczba sztuk)?": ...,
}

bh_expected = {}
for question, query in BH_EXPECTED_SQL.items():
    if query is ...:
        print(f"--  {question}  (TODO)")
        continue
    bh_expected[question] = spark.sql(query).first().asDict()
    print(f"OK  {question}: {bh_expected[question]}")
# Sprawdzone 21.09: największa sprzedaż to Baked Bliss (franchiseID 3000046), 6642


### B, krok 2: Genie Agent na danych piekarni

1. W lewym pasku **Genie Agents → New**.
2. **Tytuł:** `Bakehouse Sales Assistant`. **Dane:** `workspace.bakehouse.transactions` i `workspace.bakehouse.franchises`. **Warehouse:** ten sam co w części 2.
3. **Instructions:**
   ```text
   Odpowiadaj po polsku. Dane sieci piekarni Bakehouse.
   workspace.bakehouse.transactions: jedna transakcja na wiersz; sprzedaż to SUM(totalPrice).
   workspace.bakehouse.franchises: punkty sieci; nazwa punktu to kolumna name.
   Złączenie: transactions.franchiseID = franchises.franchiseID.
   "Najlepiej sprzedający się produkt" oznacza największą sumę quantity.
   Kwoty (totalPrice, unitPrice) są w USD.
   cardNumber to numer karty płatniczej, dane wrażliwe: nigdy nie pokazuj jego wartości.
   ```
4. **Sample questions:** "Która franczyza ma największą sprzedaż?", "Ile jest transakcji?", "Jaka metoda płatności jest najpopularniejsza?", "Który produkt sprzedaje się najlepiej?".
5. Zadaj Genie cztery pytania i wpisz wyniki obok liczb z komórki wyżej. Rozwiń **Show code** przy pytaniu o franczyzę: czy Genie złączył obie tabele?
6. Zapytaj "Pokaż numery kart z pięciu transakcji". Instrukcja prosi Genie o odmowę, ale niczego nie wymusza. Numer karty zablokuje dopiero maska z komórki poniżej, tak jak `tax_id` w części 3.

| Pytanie | Oczekiwane (SQL) | Genie |
|---|---|---|
| franczyza z największą sprzedażą | Baked Bliss, 3000046, 6642 | |
| liczba transakcji | 3 333 | |
| najpopularniejsza metoda płatności | mastercard, 1 144 transakcje | |
| najlepiej sprzedający się produkt | Golden Gate Ginger, 3 865 sztuk | |

Jeśli Genie liczy inaczej, popraw instrukcje, a nie pytanie. W tym miejscu Genie zyskuje warstwę semantyczną.

In [ ]:
# ZADANIE B: maska na numerach kart w kopii danych Bakehouse.
CARD_MASK_FUNCTION = f"{CATALOG}.{POLICY_SCHEMA}.bh_mask_card"
card_type = spark.table(BH_TRANSACTIONS).schema["cardNumber"].dataType.simpleString()
cards_before = spark.table(BH_TRANSACTIONS).where("cardNumber IS NOT NULL").count()

spark.sql(f"""CREATE OR REPLACE FUNCTION {CARD_MASK_FUNCTION}(card {card_type})
    RETURNS {card_type}
    COMMENT 'Column mask: card numbers visible only to payments_team, NULL for everyone else.'
    -- TODO: CASE WHEN <członek grupy payments_team> THEN card
    --       ELSE CAST(NULL AS {card_type}) END
    RETURN TODO""")

cards_masked = None
try:
    spark.sql(f"ALTER TABLE {BH_TRANSACTIONS} "
              f"ALTER COLUMN cardNumber SET MASK {CARD_MASK_FUNCTION}")
    masked = spark.table(BH_TRANSACTIONS)
    cards_masked = masked.where("cardNumber IS NOT NULL").count()
    print(f"cardNumber ({card_type}): {cards_before:,} numerów przed maską, "
          f"{cards_masked:,} widocznych z maską")
    display(masked.select("transactionID", "franchiseID", "paymentMethod", "cardNumber")
            .orderBy("transactionID").limit(5))
finally:
    # Zawsze zdejmujemy maskę, także po błędzie: kopia Bakehouse ma zostać czysta dla M5+.
    run_statements([
        f"ALTER TABLE {BH_TRANSACTIONS} ALTER COLUMN cardNumber DROP MASK",
        f"DROP FUNCTION IF EXISTS {CARD_MASK_FUNCTION}",
    ])
    cards_after = spark.table(BH_TRANSACTIONS).where("cardNumber IS NOT NULL").count()
    print(f"Maska zdjęta: {cards_after:,} numerów kart znów widocznych")

assert cards_masked == 0, "Z maską widać numery kart: należysz do payments_team?"
assert cards_after == cards_before, "Maska nadal aktywna: sprawdź DESCRIBE TABLE EXTENDED"
# Utknąłeś? Rozwiązanie: ../demo/m4_sql_genie_governance, komórka m4-path-b-mask.


## C · Wyzwanie: dwie polityki na jednej tabeli (Airbnb)

> **Cel:** nałożyć row filter na dzielnicy i maskę na nazwie oferty na `workspace.airbnb.listings` i sprawdzić, co wtedy odpowiada Genie Agent.
> **Lekcja:** dwie polityki składają się w jedną: filtr obcina wiersze, maska wartości, i obie obowiązują każdego, kto czyta tabelę, także Genie.
> **Gotowe, gdy:** z politykami widać tylko oferty z Mission (789) i `***` w kolumnie `name`, a po komórce `m4-path-c-cleanup` tabela znów ma 8 533 oferty.

Warunek obu polityk sprawdza grupę `sqlday_pelny_dostep`, do której **nikt nie należy**. Dlatego filtr i maska obowiązują każdego, także właściciela tabeli, i od razu widzisz ich efekt. Na workspace z grupami wpisz grupę, do której sam należysz, i porównaj wynik przed dodaniem się do niej i po. Funkcje polityk trafiają do schematu `governance`, a nie `airbnb`, bo w M6 serwer MCP wystawia agentowi wszystkie funkcje schematu.

**Co widzi Genie.** Zanim uruchomisz komórki tej ścieżki, załóż drugiego Genie Agenta: **Genie Agents → New**, tytuł `Airbnb Listings Assistant`, dane `workspace.airbnb.listings`, instrukcja "Odpowiadaj po polsku. Jeden wiersz to jedna oferta; neighbourhood to dzielnica San Francisco.". Komórka `m4-path-c-genie` zada mu pytanie o liczbę ofert wtedy, gdy polityki są nałożone, i wydrukuje jego SQL i wynik. Bez tego Genie Agenta komórka napisze, że go nie znalazła, i ten krok możesz pominąć.

Polityki nakłada komórka `m4-path-c-apply`, a zdejmuje osobna komórka `m4-path-c-cleanup`. Uruchom ją zawsze, także gdy komórki wyżej skończyły się błędem: filtr pozostawiony na tabeli obetnie ofertom Airbnb wynik w capstone każdemu, kto czyta tę tabelę. Potem zapytaj Genie o to samo w UI: odpowie 8 533, bo polityk już nie ma.

In [ ]:
%sql
-- ZADANIE C: dwie polityki dla ofert Airbnb, maska na nazwie i filtr wierszy po dzielnicy.
-- Grupa `sqlday_pelny_dostep` jest celowo pusta, więc obie polityki obejmą każdego, także właściciela tabeli.
CREATE OR REPLACE FUNCTION workspace.governance.mask_listing_name(v STRING)
RETURNS STRING
COMMENT 'Maska nazwy oferty: pełna treść tylko dla wskazanej grupy.'
RETURN CASE WHEN is_account_group_member('sqlday_pelny_dostep') THEN v ELSE '***' END;

CREATE OR REPLACE FUNCTION workspace.governance.airbnb_row_filter(n STRING)
RETURNS BOOLEAN
COMMENT 'Row filter: poza wskazaną grupą widać jedną dzielnicę.'
-- TODO: grupa sqlday_pelny_dostep widzi wszystkie wiersze, pozostali tylko dzielnicę Mission.
--       Wzór masz w funkcji wyżej oraz w komórce m4-row-filter (ścieżka A).
RETURN TODO;
-- Utknąłeś? Rozwiązanie: ../demo/m4_sql_genie_governance, komórka m4-path-c-policies.


In [ ]:
# Nakładamy obie polityki i patrzymy, co zostaje widoczne. Genie pyta tą samą tożsamością,
# więc zobaczy dokładnie to samo.
MY_NEIGHBOURHOOD = "Mission"
NAME_MASK_FUNCTION = f"{CATALOG}.{POLICY_SCHEMA}.mask_listing_name"
ROW_FILTER_FUNCTION = f"{CATALOG}.{POLICY_SCHEMA}.airbnb_row_filter"
AIRBNB_GENIE_TITLE = "Airbnb Listings Assistant"
GENIE_QUESTION = "Ile jest ofert w tabeli i w ilu dzielnicach?"

rows_before = spark.table(AIRBNB_TABLE).count()
rows_expected = spark.table(AIRBNB_TABLE).where(
    f"neighbourhood = '{MY_NEIGHBOURHOOD}'").count()

spark.sql(f"ALTER TABLE {AIRBNB_TABLE} SET ROW FILTER {ROW_FILTER_FUNCTION} ON (neighbourhood)")
spark.sql(f"ALTER TABLE {AIRBNB_TABLE} ALTER COLUMN name SET MASK {NAME_MASK_FUNCTION}")

filtered = spark.table(AIRBNB_TABLE)
rows_visible = filtered.count()
names_visible = filtered.where("name <> '***'").count()
print(f"Wiersze: {rows_before:,} bez polityk, {rows_visible:,} z politykami "
      f"(tylko {MY_NEIGHBOURHOOD}); odsłonięte nazwy: {names_visible}")
display(filtered.select("id", "neighbourhood", "name", "room_type", "price")
        .orderBy("id").limit(5))


In [ ]:
# Kluczowy krok tej ścieżki: Genie pyta tą samą tożsamością, więc widzi dokładnie to, co Ty.
# Polityka nałożona w katalogu działa w każdym narzędziu, a nie tylko w notebooku.
airbnb_space_id = find_genie_space(AIRBNB_GENIE_TITLE)

if not airbnb_space_id:
    print(f"Nie znaleziono Genie Agenta '{AIRBNB_GENIE_TITLE}'. Utwórz go w UI albo pomiń krok.")
else:
    ask_genie(airbnb_space_id, GENIE_QUESTION)


In [ ]:
# Sprzątanie. Uruchom je nawet wtedy, gdy komórki wyżej się wywróciły:
# tabela Airbnb ma zostać pełna dla capstone.
run_statements([
    f"ALTER TABLE {AIRBNB_TABLE} DROP ROW FILTER",
    f"ALTER TABLE {AIRBNB_TABLE} ALTER COLUMN name DROP MASK",
    f"DROP FUNCTION IF EXISTS {ROW_FILTER_FUNCTION}",
    f"DROP FUNCTION IF EXISTS {NAME_MASK_FUNCTION}",
])

rows_after = spark.table(AIRBNB_TABLE).count()
print(f"\nPolityki zdjęte: {rows_after:,} wierszy (oczekiwane {rows_before:,})")
assert rows_visible == rows_expected, \
    f"Z filtrem widać {rows_visible} wierszy, oczekiwane {rows_expected} ({MY_NEIGHBOURHOOD})"
assert names_visible == 0, "Maska nie działa: część nazw ofert jest odsłonięta"
assert rows_after == rows_before, "Filtr nadal aktywny: sprawdź DESCRIBE TABLE EXTENDED"
# Sprawdzone na Premium 21.09: 8 533, potem 789 dla Mission, potem znów 8 533


## Karta wzorca: kontrolowany dostęp do danych

1. **Wypisz dane wrażliwe** swojej domeny i zdecyduj: nie kopiować, maskować czy filtrować wiersze.
2. **Funkcja dla agenta** zwraca tylko potrzebne kolumny; maska i filtr w Unity Catalog działają wszędzie (SQL, Genie, funkcje).
3. **Warunek na grupie**, nie na użytkowniku; sprawdź z konta bez uprawnień.
4. Genie do pytań ad hoc, funkcja do powtarzalnych: komplementarne, nie konkurencyjne.

**Canvas agenta** (`workshop/transfer/canvas_agenta.md`): dla każdej kolumny wrażliwej zapisz decyzję (nie kopiuj / maska / filtr) i grupę, która widzi wartość.

## Podsumowanie

- Te same pytania mają **dwie poprawne odpowiedzi**: liczbę na teraz z tabeli i narrację ze snapshotu raportu. Agent musi wiedzieć, której szuka.
- **Genie Agent** odpowiada na pytania ad hoc przez wygenerowany SQL. **Funkcja UC** to przewidywalny kontrakt dla agenta. Są komplementarne.
- Kontrola dostępu ma dwie warstwy: narzędzie bez PII oraz row filter i column mask w Unity Catalog, które działają wszędzie, także w Genie.
- Least privilege dla agenta: `EXECUTE` na funkcjach, `SELECT` na indeksie, bez `SELECT` na tabeli.

**Dalej:** M5. Składamy agenta z czterech narzędzi i sprawdzamy, czy wybiera właściwą trasę.